# Task A Run 32 -- Gemma-4-12B: learning rate 2e-4

**What Task A already knows** (1,079-row holdout of train + released validation,
`815110ff24`):

| arm | macro-F1 |
|---|---|
| char SVM | 0.8117 |
| SVM 0.57 + MuRIL 0.43 (the MuRIL-era best) | 0.8155 |
| **Gemma-4-12B QLoRA, 3 epochs, seed 42 (Run 28)** | **0.8563**, +0.043 over the SVM, CI [+0.021, +0.067] |
| Gemma + SVM, 50/50 | 0.8471, -0.007 against Gemma alone |

Run 28's seed 43 collapsed: every row came out Non-Hate. The classifier now checks after
epoch 1 and reruns a collapsed job with another seed. Seed 42's holdout curve was still
rising at the last epoch: 0.817, 0.846, 0.856.

**This run** trains Gemma-4-12B QLoRA, 3 epochs unless stated otherwise, seeds 42 and 43, on the 6,114 training rows. Every arm is scored on the same holdout against `base`, which is Run 28's recipe:

| arm | change | why |
|---|---|---|
| `lr2e-4` | `--lr 2e-4` | 2e-4 is the usual QLoRA rate. Every Gemma run so far, on both tasks, was still improving at its last epoch at 1e-4 |

**How each arm is judged.** Paired bootstrap against `base`. A flag is printed `True` for the final recipe (Task A Run 33) when P(better) >= 0.7.

**The base is reused when available.** If an earlier Task A Gemma output is attached (Run 28, or any run of this line), its healthy holdout probabilities are reused. Otherwise the base trains here.

**Time:** about 5 h standalone, about 2.5 h with a base attached.

**Settings:** GPU T4 x2, Internet on. No token and no attached input are needed.
Save & Run All. A hard stop at 11 h means the notebook always finishes and packages
what it has.

In [ ]:
import json, os, pathlib, subprocess, sys, time
from fnmatch import fnmatch
T_START = time.time()
HARD_STOP = T_START + 11 * 3600   # Kaggle kills at 12 h and then saves nothing

REPO = "https://github.com/robinpnalex/Hastika-ICON2026.git"
BRANCH = "task-b"
WORK = "/tmp/hastika"             # outside /kaggle/working: the output holds results only


def git(*args):
    return subprocess.run(["git", "-C", WORK, *args], check=True,
                          capture_output=True, text=True).stdout.strip()


try:
    remote = subprocess.run(["git", "ls-remote", "--exit-code", REPO, f"refs/heads/{BRANCH}"],
                            check=True, capture_output=True, text=True,
                            timeout=60).stdout.split()[0]
except Exception as e:
    raise SystemExit("Cannot reach GitHub. Turn on Settings -> Internet, then restart.") from e
if os.path.isdir(WORK + "/.git"):
    git("fetch", "-q", "--depth", "1", "origin", BRANCH)
    git("reset", "-q", "--hard", "FETCH_HEAD")
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "--depth", "1", REPO, WORK], check=True)
head = git("rev-parse", "HEAD")
assert head == remote, f"clone is at {head[:8]} but {BRANCH} is at {remote[:8]}"
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
print("repo at", git("log", "-1", "--oneline"))
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass                              # Gemma-4 is not gated; no token is needed

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score

assert torch.cuda.is_available(), "select GPU T4 x2"
N_GPU = torch.cuda.device_count()
print("gpus:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
OUT = pathlib.Path("/kaggle/working/a32_outputs")
RUNS, LOGS, SUB = OUT / "runs", OUT / "logs", OUT / "submissions"
for p in (RUNS, LOGS, SUB):
    p.mkdir(parents=True, exist_ok=True)
HF_CACHE = "/tmp/hf"
os.environ["HF_HUB_CACHE"] = HF_CACHE

from hastika.common.gpu_queue import run_queue
from hastika.task_b.combined_holdout import fingerprint, paired_bootstrap

TASK = "a"
CLASSES = ["Non-Hate", "Hate"]
LABEL_COL = "Label"
SPLIT = pathlib.Path(WORK) / "data/derived/task_a_combined_holdout"
df = pd.read_csv(SPLIT / "all.csv")
holdout = pd.read_csv(SPLIT / "holdout.csv")
assert (len(df), len(holdout)) == (7193, 1079), (len(df), len(holdout))
HOLDOUT_FP = fingerprint(holdout["id"])
assert HOLDOUT_FP == "815110ff24", HOLDOUT_FP
y_ho = holdout[LABEL_COL].map(CLASSES.index).to_numpy()
TEST_CSV = pathlib.Path("/tmp/test_inputs.csv")
pd.read_csv(pathlib.Path(WORK) / "data/raw/hastika_binary_test.csv")[["id", "Comment"]].to_csv(TEST_CSV, index=False)
test = pd.read_csv(TEST_CSV)
EXTERNAL = pathlib.Path(WORK) / "data/external/offenseval_kn.csv"
print(f"task {TASK.upper()}: train {len(df) - len(holdout)}, holdout {len(holdout)} ({HOLDOUT_FP}), "
      f"all {len(df)}, test {len(test)}")


def report(y, probs, name):
    pred = probs.argmax(1)
    per = f1_score(y, pred, average=None, labels=range(len(CLASSES)), zero_division=0)
    macro = float(f1_score(y, pred, average="macro"))
    print(f"{name:34s} macro-F1 {macro:.4f}  acc {accuracy_score(y, pred):.4f}  | "
          + "  ".join(f"{c[:5]} {f:.3f}" for c, f in zip(CLASSES, per)))
    return macro


def boot_line(label, a, b):
    r = paired_bootstrap(y_ho, a, b)
    print(f"{label:28s} {r['diff']:+.4f}  95% CI [{r['ci95'][0]:+.4f}, {r['ci95'][1]:+.4f}]  "
          f"P(better) {r['p_better']:.2f}")
    return r


# Earlier runs' outputs are reused when attached (Add Input -> Notebook Output), found by
# walking every input with symlinks followed. Nothing below *requires* them.
def find_prior(pattern):
    hits = []
    for root, _, files in os.walk("/kaggle/input", followlinks=True):
        for f in files:
            p = os.path.join(root, f)
            if fnmatch(p, pattern):
                hits.append(pathlib.Path(p))
    return sorted(hits)


def show_inputs():
    if not os.path.isdir("/kaggle/input"):
        print("no /kaggle/input")
        return
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        depth = root[len("/kaggle/input"):].count(os.sep)
        if depth <= 4:
            print("  " * depth + (os.path.basename(root) or "/kaggle/input") + f"/  ({len(files)} files)")


print("attached inputs:")
show_inputs()

In [ ]:
# LLM stack in its own environment (transformers >= 5.12 for Gemma-4); Kaggle's torch and
# CUDA are reused via --system-site-packages. The repo's MuRIL/SVM code stays on the
# system environment.
LLM_PY = "/tmp/llmenv/bin/python"
if not os.path.exists(LLM_PY):
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip",
                    "/tmp/llmenv"], check=True)
subprocess.run(f'{LLM_PY} -m pip install -q "transformers>=5.12,<6" "peft>=0.17" '
               f'"bitsandbytes>=0.46" "accelerate>=1.0" emoji ftfy sentencepiece protobuf',
               shell=True, check=True)
subprocess.run([LLM_PY, "-c", "import transformers, peft, bitsandbytes; print('LLM env: "
                "transformers', transformers.__version__, '| peft', peft.__version__)"], check=True)

MODEL = "google/gemma-4-12B"
S = "gemma-4-12b"


def download(model=MODEL):
    from huggingface_hub import snapshot_download
    snapshot_download(model, cache_dir=HF_CACHE, token=os.environ.get("HF_TOKEN") or None,
                      allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt",
                                      "tokenizer*", "*.jinja"])


def llm_job(name, train, predict, seed, evalf="", extra=""):
    ev = f"--eval {evalf}" if evalf else ""
    return {"name": name, "log": str(LOGS / f"{name}.log"),
            "cmd": (f"{LLM_PY} -u -m hastika.task_b.llm_classifier --task {TASK} --model {MODEL} "
                    f"--train {train} {ev} --predict {' '.join(map(str, predict))} "
                    f"--out {RUNS / name} --seed {seed} --cache {HF_CACHE} {extra}")}


def tapt_job(name, texts):
    """Gemma TAPT: LoRA next-token training on unlabelled comments (the MuRIL +2.9 lesson)."""
    return {"name": name, "log": str(LOGS / f"{name}.log"),
            "cmd": (f"{LLM_PY} -u -m hastika.task_b.llm_tapt --model {MODEL} "
                    f"--texts {' '.join(map(str, texts))} --out {RUNS / name} --cache {HF_CACHE}")}


def load_probs(run_dir, csv):
    """<input stem>_probs.npy, or None when it is missing, belongs to other rows (another
    task's output found among the inputs), or is collapsed onto one class (Task A Run 28's
    seed 43 predicted Non-Hate for every row)."""
    run_dir, csv = pathlib.Path(run_dir), pathlib.Path(csv)
    p = run_dir / f"{csv.stem}_probs.npy"
    if not p.exists():
        return None
    ids = pd.read_csv(run_dir / f"{csv.stem}_ids.csv")["id"].tolist()
    if ids != pd.read_csv(csv)["id"].tolist():
        return None
    probs = np.load(p)
    share = np.bincount(probs.argmax(1), minlength=probs.shape[1]) / len(probs)
    if share.max() > 0.9:
        print(f"skipping {run_dir.name}: collapsed, {share.max():.0%} of rows in one class")
        return None
    return probs


def run_jobs(jobs, on_done=None):
    """run_queue with two automatic retries per job: exit code 4 (collapsed after epoch 1)
    reruns with seed + 1000; out of GPU memory reruns at micro-batch 4 (same effective
    batch). on_done sees each job's final outcome."""
    import re
    registry, retried = {j["name"]: j for j in jobs}, set()

    def done(name, code):
        job, nxt = registry.get(name), []
        if code and job is not None and name not in retried:
            log = open(job["log"], errors="replace").read()
            seed = re.search(r"--seed (\d+)", job["cmd"])
            if code == 4 and seed:
                print(f"{name} collapsed -- rerunning with seed {int(seed.group(1)) + 1000}")
                nxt = [{**job, "cmd": job["cmd"] + f" --seed {int(seed.group(1)) + 1000}"}]
            elif "OutOfMemoryError" in log and "llm_tapt" in job["cmd"]:
                # TAPT: same effective batch (16) at micro-batch 1, smaller logit chunks
                print(f"{name} ran out of GPU memory -- rerunning TAPT at micro-batch 1")
                nxt = [{**job, "cmd": job["cmd"] + " --bs 1 --grad-accum 16 --logit-chunk 16"}]
            elif "OutOfMemoryError" in log:
                print(f"{name} ran out of GPU memory -- rerunning at micro-batch 4")
                nxt = [{**job, "cmd": job["cmd"] + " --bs 4 --grad-accum 4 --eval-bs 8"}]
            if nxt:
                retried.add(name)
        if not nxt and on_done is not None:
            nxt = list(on_done(name, code) or [])
        for j in nxt:
            registry[j["name"]] = j
        return nxt

    return run_queue(jobs, N_GPU, raise_on_fail=False, on_done=done, stop_at=HARD_STOP)


def write_zip(name, probs, recommended=False):
    import zipfile
    assert probs.shape == (len(test), len(CLASSES)), probs.shape
    pred = pd.DataFrame({"id": test["id"], "label": [CLASSES[i] for i in probs.argmax(1)]})
    assert pred["id"].is_unique and set(pred["label"]) <= set(CLASSES)
    np.save(SUB / f"{name}_test_probs.npy", probs)
    csv = SUB / f"predictions_{name}.csv"
    pred.to_csv(csv, index=False)
    path = SUB / f"{'RECOMMENDED_' if recommended else ''}{name}.zip"
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(csv, "predictions.csv")
    prior = df[LABEL_COL].value_counts(normalize=True)
    counts = pred["label"].value_counts(normalize=True)
    print(f"{path.name}  predicted/prior: "
          + "  ".join(f"{c[:5]} {counts.get(c, 0):.1%}/{prior[c]:.1%}" for c in CLASSES))
    return path


download()

In [ ]:
base = {}
for s in (42, 43):
    for pat in (f"*/runs/base_s{s}/holdout_probs.npy", f"*/runs/ho_s{s}/holdout_probs.npy"):
        got = [load_probs(h.parent, SPLIT / "holdout.csv") for h in find_prior(pat)]
        got = [g for g in got if g is not None]
        if got:
            base[s] = got[0]
            break
print("base reused for seeds:", list(base))

ARMS = {'lr2e-4': '--lr 2e-4'}
HO = dict(train=SPLIT / "train.csv", predict=[SPLIT / "holdout.csv"], evalf=SPLIT / "holdout.csv")
TAPT_DIR = RUNS / "tapt_train"
jobs = [llm_job(f"{a}_s{s}", seed=s, extra=x, **HO) for a, x in ARMS.items() if x for s in (42, 43)]
jobs += [llm_job(f"base_s{s}", seed=s, **HO) for s in (42, 43) if s not in base]
USE_TAPT = False
if USE_TAPT:
    jobs.insert(0, tapt_job("tapt_train", [SPLIT / "train.csv", EXTERNAL]))


def on_done(name, code):
    if name == "tapt_train":
        if code == 0:
            return [llm_job(f"tapt_s{s}", seed=s, extra=f"--init-adapter {TAPT_DIR}", **HO)
                    for s in (42, 43)]
        print("TAPT failed -- the tapt arm is skipped, see", LOGS / "tapt_train.log")
    return []


codes = run_jobs(jobs, on_done)
print("exit codes:", codes, f"| {(time.time() - T_START) / 3600:.1f} h")

In [ ]:
def arm(prefix):
    got = {s: load_probs(RUNS / f"{prefix}_s{s}", SPLIT / "holdout.csv") for s in (42, 43)}
    return {s: p for s, p in got.items() if p is not None}


results = {"base": base or arm("base"), **{a: arm(a) for a in ARMS}}
avg, scores, boots = {}, {}, {}
for name, seeds in results.items():
    for s, p in seeds.items():
        report(y_ho, p, f"{name} seed {s}")
        m = RUNS / f"{name}_s{s}" / "metrics.json"
        if m.exists():
            print("   per epoch (diagnostic):",
                  [round(h.get("eval_macro_f1", 0), 4) for h in json.load(open(m))["history"]])
    if seeds:
        avg[name] = np.mean(list(seeds.values()), 0)
        scores[name] = report(y_ho, avg[name], f"{name}, {len(seeds)} seed(s)")
print()
for a in ARMS:
    if a in avg and "base" in avg:
        boots[a] = boot_line(f"{a} - base", avg[a], avg["base"])
USE = {k: v["p_better"] >= 0.7 for k, v in boots.items()}
print("\nflags for the final recipe (P(better than base) >= 0.7):", USE)
json.dump({"commit": head, "holdout_scores": scores, "bootstrap": boots, "use": USE,
           "exit_codes": codes}, open(OUT / "result.json", "w"), indent=2)